# Summarize Data

In [9]:
# connect to Enterprise GIS
from arcgis.gis import GIS
import arcgis.geoanalytics

portal_gis = GIS("https://ndhwks6.esri.com/portal", "admin", 'esri.agp', verify_cert=False)

In [10]:
search_result = portal_gis.content.search("bigDataFileShares_all_hurricanes", item_type = "big data file share")[0]
search_result

<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>

In [11]:
years_50 = search_result.layers[0]

In [12]:
search_result = portal_gis.content.search("bigDataFileShares_ServiceCallsOrleans", item_type = "big data file share")[0]
search_result

<Item title:"bigDataFileShares_ServiceCallsOrleans" type:Big Data File Share owner:admin>

In [13]:
calls = search_result.layers[0]

## Aggregate Points

In [14]:
from arcgis.geoanalytics.summarize_data import aggregate_points

In [15]:
output1 = aggregate_points(point_layer=calls, #the input point layer to be aggregated
                          bin_type='Hexagon', #type of bins to be created
                          bin_size=1, # size of bin 
                          bin_size_unit='Meters', # unit of bin
                          output_name='aggregate output') # output name for aggregated result

{"messageCode":"BD_101051","message":"Possible issues were found while reading 'pointLayer'.","params":{"paramName":"pointLayer"}}
{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}
{"messageCode":"BD_101054","message":"Some records have either missing or invalid geometries."}


## Build Multi-Variable Grid

In [16]:
from arcgis.geoanalytics.summarize_data import build_multivariable_grid

In [17]:
var_calc = [{"layer":0,"variables":[{"type":"AttributeOfNearest","outFieldName":"test","attributeField":"Location","searchDistance":6,"searchDistanceUnit":"Miles"}]}]

In [18]:
##usage example
output = build_multivariable_grid(input_layers=[calls], 
                                  variable_calculations=var_calc, 
                                  bin_size=5, 
                                  bin_unit='Miles', 
                                  bin_type='Square', 
                                  output_name='build_multivariable_grid')
output

<Item title:"build_multivariable_grid" type:Feature Layer Collection owner:admin>

## Describe dataset

In [19]:
from arcgis.geoanalytics.summarize_data import describe_dataset
from datetime import datetime as dt

In [20]:
description = describe_dataset(input_layer=calls,
                               extent_output=True,
                               sample_size=1000,
                               output_name="Description of service calls" + str(dt.now().microsecond),
                               return_tuple=True)

{"messageCode":"BD_101051","message":"Possible issues were found while reading 'inputLayer'.","params":{"paramName":"inputLayer"}}
{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}
{"messageCode":"BD_101054","message":"Some records have either missing or invalid geometries."}


In [21]:
description.sample_layer

<FeatureLayer url:"https://ndhwks6.esri.com/server/rest/services/Hosted/Description_of_service_calls923768/FeatureServer/2">

In [22]:
description.output

<FeatureLayer url:"https://ndhwks6.esri.com/server/rest/services/Hosted/Description_of_service_calls923768/FeatureServer/0">

In [23]:
description.output_json

{'datasetName': 'calls',
 'datasetSource': 'Big Data File Share - ServiceCallsOrleans',
 'recordCount': 3952898,
 'geometry': {'geometryType': 'Point',
  'sref': {'wkid': 102682, 'latestWkid': 3452},
  'countNonEmpty': 1763656,
  'countEmpty': 2189242,
  'spatialExtent': {'xmin': 0,
   'ymin': 0,
   'xmax': 37369000.0,
   'ymax': 3513814}},
 'time': {'timeType': 'Instant',
  'countNonEmpty': 1763656,
  'countEmpty': 2189242,
  'temporalExtent': {'start': '2011-01-01 00:00:02.000',
   'end': '2019-08-03 23:59:09.000'}}}

## Reconstruct tracks

In [24]:
from arcgis.geoanalytics.summarize_data import reconstruct_tracks

In [25]:
##usage example
result = reconstruct_tracks(years_50,
                            track_fields='Serial_Num',
                            method='GEODESIC')

{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}


In [26]:
result

<Item title:"Reconstruct_Tracks_I7YC1Q" type:Feature Layer Collection owner:admin>

## Summarize attributes

In [27]:
from arcgis.geoanalytics.summarize_data import summarize_attributes

In [28]:
summarized_features = summarize_attributes(input_layer=years_50,
                                           fields='track_type')
summarized_features

{"messageCode":"BD_101051","message":"Possible issues were found while reading 'inputLayer'.","params":{"paramName":"inputLayer"}}
{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}


<Item title:"Summarize_Attributes_YKJQY6" type:Feature Layer Collection owner:admin>

## Summarize center and dispersion

In [29]:
from arcgis.geoanalytics.summarize_data import summarize_center_and_dispersion

In [30]:
output_trends = summarize_center_and_dispersion(input_layer=years_50, 
                                                summary_type='MedianCenter', 
                                                output_name='directional trends')
output_trends

{"messageCode":"BD_101051","message":"Possible issues were found while reading 'inputLayer'.","params":{"paramName":"inputLayer"}}
{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}


<Item title:"directional_trends" type:Feature Layer Collection owner:admin>

## Summarize within

In [31]:
from arcgis.geoanalytics.summarize_data import summarize_within

In [32]:
##usage example
summarised_features = summarize_within(years_50, 
                                       bin_type="Square",
                                       bin_size=5,
                                       bin_size_unit='Miles',
                                       standard_summary_fields=[{"statisticType" : "average", "onStatisticField" : "Wind" }],
                                       output_name='summmrized_features')
summarised_features

{"messageCode":"BD_101137","message":"Bin generation and analysis requires a projected coordinate system. The Equal Earth projection has been applied with custom projection parameters based on the analysis extent.","params":{"projectionName":"Equal Earth"}}
{"messageCode":"BD_101051","message":"Possible issues were found while reading 'summarizedLayer'.","params":{"paramName":"summarizedLayer"}}
{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}


<Item title:"summmrized_features" type:Feature Layer Collection owner:admin>